# Pegasus PubMed

## Introduction
This notebook demonstrates **Pegasus PubMed** for text summarization. This pretrained checkpoint generates a domain-adapted abstractive summary.

### Objectives
- Load the shared article from `article.txt`.
- Generate a summary with this notebook's method.
- Evaluate the result against `reference_summary.txt` with shared ROUGE and BLEU metrics.

### Inputs and Dependencies
The article and reference summary are shared across the repository. Classical methods use the dependencies listed in `requirements.txt`; pretrained inference also downloads its model checkpoint.


In [1]:
import os, sys
from pathlib import Path
CURRENT_DIR = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
     if (candidate / "summarization_utils.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate the repository root containing summarization_utils.py")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("HF_HOME", str(ROOT / ".model_cache"))
os.environ.setdefault("NLTK_DATA", str(ROOT / ".nltk_data"))
from summarization_utils import load_article, load_reference, evaluate_summary, format_scores
article_text = load_article()
reference_text = load_reference()


**Pegasus-PubMed** is a pre-trained language model introduced in 2020 for the task of summarizing biomedical literature from the PubMed database. It is based on the Pegasus architecture and is trained on a large and diverse dataset of biomedical literature covering various fields, including genetics, pharmacology, microbiology, and more. Pegasus-PubMed uses a transformer-based sequence-to-sequence model that is capable of generating high-quality summaries that capture the most important information from the original biomedical literature. The model has achieved state-of-the-art performance on the PubMed dataset and has also shown promising results on other summarization tasks related to biomedical literature. Pegasus-PubMed is available as part of the Hugging Face Transformers library, making it easy to use for natural language processing tasks related to biomedical literature.

In [2]:
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    from summarization_utils import load_article, load_reference, evaluate_summary, format_scores

    article = load_article()
    reference_summary = load_reference()
    model_name = 'google/pegasus-pubmed'
    # The MultiNews checkpoint expects document separators; all other checkpoints receive the article as one document.
    model_input = article
    if model_name.endswith("multi_news"):
        model_input = " ||| ".join(part.strip() for part in article.split("\n\n") if part.strip())

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    max_input_tokens = max(1, min(1024, getattr(model.config, "max_position_embeddings", 1024) - 2))
    inputs = tokenizer(model_input, truncation=True, max_length=max_input_tokens, return_tensors="pt")
    with torch.inference_mode():
        generated = model.generate(**inputs, num_beams=4, max_length=120, min_length=20, no_repeat_ngram_size=3)
    summary = tokenizer.decode(generated[0], skip_special_tokens=True)
    print("Summary:", summary)
    print("Metrics:", format_scores(evaluate_summary(summary, reference_summary)))
except (OSError, RuntimeError, ImportError) as error:
    summary = None
    print(f"Pegasus inference unavailable in this environment: {type(error).__name__}: {error}")


C:\Users\tuhin\Downloads\Text-Summarization-Models-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-pubmed and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Summary: global climate data for the 20th century were analyzed to show that 2015 was the second warmest on record for the globe , second only to 1998 . <n> global warming was the third strongest on record since recordkeeping began in the late 19th century . in this article , the authors present their views on how global warming is affecting the way people live , work , and play.imagesfigure 1.figure 2.figure 3.figure 4.figure 5.figure 6.figure 7.figure 8.figure 9.figure 10.figure 11.figure 12.figure 13.
Metrics: rouge1_f1=0.176, rouge2_f1=0.038, rougeL_f1=0.101, bleu4=0.037
